# Rothermel surface-fire rate of spread
## Scott & Burgan (2005) standard fire behavior fuel models

Pick one of the 40 standard fuel models, set the environmental inputs
(fuel moisture, midflame wind, slope), and read off the **rate of spread (ROS)**.

The calculation runs in Rothermel's original *imperial* units so the intermediate
parameters (characteristic SAV, packing ratio, reaction intensity, ...) match the
published tables; the final ROS is converted to metric at the end.

**References**

* Rothermel, R.C. 1972. *A mathematical model for predicting fire spread in wildland fuels.* INT-115.
* Albini, F.A. 1976. *Estimating wildfire behavior and effects.* INT-GTR-30.
* Scott, J.H.; Burgan, R.E. 2005. *Standard fire behavior fuel models.* RMRS-GTR-153.
  (the "FBFM40" set; frequently mis-cited as *Scott & Burgess*).
* Andrews, P.L. 2018. *The Rothermel surface fire spread model ...* RMRS-GTR-371. (equation listing)

> **Note** Four transcription errors in the source fuel-model JSON are corrected in
> `rothermel.py` (`_CORRECTIONS`): SH1/SH6/TL3 100-h loads and the SH9 fuel-bed depth.
> The implementation reproduces the published characteristic SAV and packing ratio for
> all 40 models (run `python validate.py`).

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

from rothermel import (
    load_fuel_models, compute_ros, Moisture,
    mps_to_ft_min, kmh_to_ft_min, deg_to_slope_fraction,
    wind_adjustment_factor, wind_10m_to_20ft,
    TONS_PER_ACRE_TO_LB_PER_FT2 as T2LB,
)

MODELS = load_fuel_models()           # 40 burnable models, keyed by code
print(f'Loaded {len(MODELS)} standard fuel models.')

## Inputs

Wind is **midflame** wind speed (the wind acting on the flames, typically a fraction
of the 10 m open wind). Slope is the terrain angle in degrees. Live herbaceous moisture
drives the *dynamic* load transfer (curing) for the grass/grass-shrub models.

In [ ]:
# ----- fuel model selector (grouped by category) -----
options = [(f'{c}  -  {MODELS[c].name}', c) for c in MODELS]
fuel_dd = widgets.Dropdown(options=options, value='GR2', description='Fuel model',
                           layout=widgets.Layout(width='520px'),
                           style={'description_width': '120px'})

def _slider(desc, lo, hi, val, step, unit):
    return widgets.FloatSlider(description=desc, min=lo, max=hi, value=val, step=step,
                               continuous_update=False, readout_format='.0f',
                               layout=widgets.Layout(width='420px'),
                               style={'description_width': '150px'})

# ----- fuel load sliders (oven-dry, t/acre) -----
# defaults are read from the selected fuel model's published table loads;
# _sync_loads below resets them whenever the fuel model changes.
_fm0 = MODELS[fuel_dd.value]

def _load_slider(desc, val):
    return widgets.FloatSlider(description=desc, min=0.0, max=12.0, value=val, step=0.05,
                               continuous_update=False, readout_format='.2f',
                               layout=widgets.Layout(width='420px'),
                               style={'description_width': '150px'})

w_1h    = _load_slider('1-h load t/ac',    _fm0.w_1h    / T2LB)
w_10h   = _load_slider('10-h load t/ac',   _fm0.w_10h   / T2LB)
w_100h  = _load_slider('100-h load t/ac',  _fm0.w_100h  / T2LB)
w_herb  = _load_slider('herb load t/ac',   _fm0.w_herb  / T2LB)
w_woody = _load_slider('woody load t/ac',  _fm0.w_woody / T2LB)

def _sync_loads(*_):
    """Reset the load sliders to the newly selected model's table defaults."""
    fm = MODELS[fuel_dd.value]
    w_1h.value    = fm.w_1h    / T2LB
    w_10h.value   = fm.w_10h   / T2LB
    w_100h.value  = fm.w_100h  / T2LB
    w_herb.value  = fm.w_herb  / T2LB
    w_woody.value = fm.w_woody / T2LB

fuel_dd.observe(_sync_loads, names='value')

# ----- live load summary (t/acre, t/hectare, total) -----
TAC_TO_THA = 2.4710538   # 1 t/acre -> t/hectare (1 ha = 2.4710538 acre)
load_table = widgets.Output()

def _render_load_table(*_):
    rows = [('1-h', w_1h.value), ('10-h', w_10h.value), ('100-h', w_100h.value),
            ('herb', w_herb.value), ('woody', w_woody.value)]
    total = sum(v for _, v in rows)
    body = '\n'.join(f'| {name} | {v:.2f} | {v*TAC_TO_THA:.2f} |' for name, v in rows)
    md = ('| Load | t/acre | t/hectare |\n'
          '|---|---:|---:|\n'
          f'{body}\n'
          f'| **Total** | **{total:.2f}** | **{total*TAC_TO_THA:.2f}** |')
    with load_table:
        load_table.clear_output(wait=True)
        display(Markdown(md))

for _w in (w_1h, w_10h, w_100h, w_herb, w_woody):
    _w.observe(_render_load_table, names='value')
_render_load_table()

m_1h    = _slider('1-h dead moist %',    1, 40,  6, 1, '%')
m_10h   = _slider('10-h dead moist %',   1, 40,  7, 1, '%')
m_100h  = _slider('100-h dead moist %',  1, 50,  8, 1, '%')
m_herb  = _slider('live herb moist %',  30, 300, 90, 5, '%')
m_woody = _slider('live woody moist %', 30, 300, 90, 5, '%')

wind    = widgets.FloatText(value=5.0, description='Wind speed',
                            style={'description_width': '150px'},
                            layout=widgets.Layout(width='300px'))
wind_unit = widgets.Dropdown(options=['m/s', 'km/h', 'mi/h', 'ft/min'], value='m/s',
                             layout=widgets.Layout(width='90px'))
# what the entered wind refers to; midflame = used directly (no WAF applied)
wind_ref = widgets.Dropdown(
    options=[('measured at 10 m (open)', '10m'),
             ('measured at 20 ft', '20ft'),
             ('midflame (use directly)', 'midflame')],
    value='10m', description='Wind reference',
    style={'description_width': '150px'}, layout=widgets.Layout(width='420px'))
slope   = _slider('slope (deg)', 0, 50, 0, 1, 'deg')
wind_limit = widgets.Checkbox(value=False, description='Apply Rothermel wind limit (0.9 I_R)',
                              indent=False)

# ----- wind adjustment factor (20-ft -> midflame) -----
waf_mode = widgets.Dropdown(
    options=[('auto from canopy (sheltered if crown-fill > 5%)', 'auto'),
             ('unsheltered (fuel-bed depth only)', 'unsheltered'),
             ('enter WAF directly', 'direct')],
    value='unsheltered', description='WAF model',
    style={'description_width': '150px'}, layout=widgets.Layout(width='480px'))
canopy_cover  = _slider('canopy cover %',   0, 100, 40, 5, '%')
canopy_height = _slider('canopy height (ft)', 0, 150, 50, 5, 'ft')
crown_ratio   = _slider('crown ratio %',    0, 100, 70, 5, '%')
waf_direct    = widgets.FloatText(value=0.30, description='WAF (direct)',
                                  style={'description_width': '150px'},
                                  layout=widgets.Layout(width='280px'))

ui = widgets.VBox([
    fuel_dd,
    widgets.HTML('<b>Fuel load</b> (oven-dry, t/acre; reset to table defaults on model change)'),
    w_1h, w_10h, w_100h, w_herb, w_woody, load_table,
    widgets.HTML('<b>Dead fuel moisture</b>'), m_1h, m_10h, m_100h,
    widgets.HTML('<b>Live fuel moisture</b>'), m_herb, m_woody,
    widgets.HTML('<b>Wind &amp; slope</b>'),
    widgets.HBox([wind, wind_unit]), wind_ref, slope, wind_limit,
    widgets.HTML('<b>Wind adjustment factor</b> (ignored if wind reference is midflame)'),
    waf_mode, canopy_cover, canopy_height, crown_ratio, waf_direct,
])

## Rate of spread
Adjust any control above and the result below updates automatically.

In [ ]:
import math
from dataclasses import replace

from burnout import compute_burnout

_WIND_CONV = {'m/s': mps_to_ft_min, 'km/h': kmh_to_ft_min,
              'mi/h': lambda v: v * 88.0, 'ft/min': lambda v: v}

# imperial -> SI conversions for the intermediate parameters
IR_BTUFT2MIN_TO_KWM2 = 0.189266    # BTU/ft^2/min -> kW/m^2  (J/m^2/s)
SAV_PERFT_TO_PERM    = 3.280840    # ft^-1 -> m^-1
HEAT_BTUFT3_TO_KJM3  = 37.2589     # BTU/ft^3 -> kJ/m^3
MS_TO_FT_MIN         = 196.850394  # 1 m/s -> ft/min
LATENT_HEAT_WATER    = 2.26e6      # latent heat of vaporisation [J/kg]

out = widgets.Output()

def _midflame(fm):
    """Return (midflame_ft_min, info_dict) applying the chosen WAF chain."""
    speed_ftmin = _WIND_CONV[wind_unit.value](wind.value)   # at its reference height
    if wind_ref.value == 'midflame':
        return speed_ftmin, {'mode': 'midflame'}
    # reduce to 20-ft open wind first
    w20 = wind_10m_to_20ft(speed_ftmin) if wind_ref.value == '10m' else speed_ftmin
    ref = '10-m' if wind_ref.value == '10m' else '20-ft'
    if waf_mode.value == 'direct':
        waf = waf_direct.value
        return w20 * waf, {'mode': 'direct', 'waf': waf, 'ref': ref}
    cc = canopy_cover.value / 100 if waf_mode.value == 'auto' else 0.0
    res = wind_adjustment_factor(fm.depth, canopy_cover=cc,
                                 canopy_height_ft=canopy_height.value,
                                 crown_ratio=crown_ratio.value / 100)
    return w20 * res.waf, {
        'mode': res.regime, 'waf': res.waf, 'ref': ref,
        'fuel_depth': fm.depth, 'canopy_height': canopy_height.value,
        'canopy_cover': cc, 'crown_ratio': crown_ratio.value / 100,
        'crown_fill': res.crown_fill,
    }

def _waf_summary(info):
    if info['mode'] == 'midflame':
        return 'entered as midflame wind (no WAF applied)'
    return f"WAF = {info['waf']:.2f} ({info['mode']}); {info['ref']} wind reduced to midflame"

def _waf_block(info):
    """Markdown block with the WAF formula and the substituted values."""
    if info['mode'] == 'midflame':
        return ''
    if info['mode'] == 'direct':
        return (f"\n**Wind adjustment factor (WAF) = {info['waf']:.2f}** "
                f"(user-entered). Midflame wind = {info['ref']} wind × WAF.\n")
    if info['mode'] == 'unsheltered':
        H = info['fuel_depth']
        log_val = math.log((20.0 + 0.36 * H) / (0.13 * H))
        return rf'''
**Wind adjustment factor** (Albini & Baughman 1979; Andrews 2012 GTR-266) — *unsheltered* (no overstory, or crown-fill $\leq 5\%$): WAF depends only on the fuel-bed depth $H_f$.

$$\text{{WAF}} = \frac{{1.83}}{{\ln\!\left(\dfrac{{20 + 0.36\,H_f}}{{0.13\,H_f}}\right)}}
= \frac{{1.83}}{{\ln\!\left(\dfrac{{20 + 0.36 \cdot {H}}}{{0.13 \cdot {H}}}\right)}}
= \frac{{1.83}}{{{log_val:.4f}}}
= {info['waf']:.3f}$$

with $H_f = {H}$ ft. Midflame wind = {info['ref']} wind × WAF.
'''
    # sheltered
    H = info['canopy_height']
    f_val = info['crown_fill']
    log_val = math.log((20.0 + 0.36 * H) / (0.13 * H))
    sqrt_val = math.sqrt(f_val * H)
    return rf'''
**Wind adjustment factor** (Albini & Baughman 1979; Andrews 2012 GTR-266) — *sheltered* (crown-fill portion $f > 5\%$): WAF uses canopy height $H_c$ and crown-fill $f = (CC/3)\,CR$.

$$\text{{WAF}} = \frac{{0.555}}{{\sqrt{{f\,H_c}}\,\ln\!\left(\dfrac{{20 + 0.36\,H_c}}{{0.13\,H_c}}\right)}}
= \frac{{0.555}}{{\sqrt{{{f_val:.3f} \cdot {H}}}\,\ln\!\left(\dfrac{{20 + 0.36 \cdot {H}}}{{0.13 \cdot {H}}}\right)}}
= \frac{{0.555}}{{{sqrt_val:.3f} \cdot {log_val:.4f}}}
= {info['waf']:.3f}$$

with $H_c = {H}$ ft, $f = (CC/3)\,CR = ({info['canopy_cover']:.2f}/3) \cdot {info['crown_ratio']:.2f} = {f_val:.3f}$. Midflame wind = {info['ref']} wind × WAF.
'''

def _render(*_):
    # start from the published model, then override the oven-dry loads with the
    # slider values (t/acre -> lb/ft^2). dynamic/static nature is kept from the model.
    fm = replace(MODELS[fuel_dd.value],
                 w_1h=w_1h.value * T2LB,
                 w_10h=w_10h.value * T2LB,
                 w_100h=w_100h.value * T2LB,
                 w_herb=w_herb.value * T2LB,
                 w_woody=w_woody.value * T2LB)
    moist = Moisture.from_percent(m_1h.value, m_10h.value, m_100h.value,
                                  m_herb.value, m_woody.value)
    wind_ftmin, waf_info = _midflame(fm)
    r = compute_ros(fm, moist, wind_ftmin, deg_to_slope_fraction(slope.value),
                    apply_wind_limit=wind_limit.value)
    # burnout / heat-release closure for the selected fuel + current moisture.
    # I_R-consistent basis: I_R already carries the moisture/mineral damping, so it
    # IS the sensible flux; the fuel water leaves as an additional latent flux on top,
    # scaled by the gross-chemistry split ratio so everything vanishes with I_R.
    b = compute_burnout(fm, moist, characteristic_sav=r.characteristic_sav,
                        reaction_intensity=r.reaction_intensity)
    # peak fluxes at ignition (t = t_i): sensible == I_R (already moisture-damped),
    # latent is the extra water-vapour flux, total is their sum
    peak_sensible_kW = b.peak_flux_kW                                   # E_net / T_f = I_R
    peak_latent_kW = b.energy_latent / b.t_f / 1e3 if b.t_f > 0 else 0.0
    peak_total_kW = b.energy_gross / b.t_f / 1e3 if b.t_f > 0 else 0.0
    water_vap_kg = b.energy_latent / LATENT_HEAT_WATER                  # I_R-consistent
    midflame_ms = wind_ftmin / MS_TO_FT_MIN

    # no-wind/no-slope rate of spread (R_0 = I_R * xi / heat_sink)
    ros_0_ft_min = r.ros_ft_min / (1.0 + r.wind_factor + r.slope_factor)
    ros_0_m_min  = ros_0_ft_min * 0.3048
    ros_0_m_s  = ros_0_ft_min / 60

    # wind-factor coefficients (Rothermel 1972; sigma in ft^-1, U in ft/min)
    sigma = r.characteristic_sav
    c_w = 7.47 * math.exp(-0.133 * sigma ** 0.55)
    b_w = 0.02526 * sigma ** 0.54
    e_w = 0.715 * math.exp(-3.59e-4 * sigma)
    # lump the sigma/packing terms:  phi_w = K * U^B  with K = C * beta_rel^(-E)
    u_factor = MS_TO_FT_MIN ** b_w          # 196.85^B, the wind-unit conversion
    k_imp = c_w * r.relative_packing_ratio ** (-e_w)   # K with U in ft/min
    k_si = k_imp * u_factor                            # K with U in m/s

    def t2(x):   # lb/ft^2 -> t/acre for display
        return x / T2LB
    loads = (f'1-h {t2(fm.w_1h):.2f}, 10-h {t2(fm.w_10h):.2f}, 100-h {t2(fm.w_100h):.2f}, '
             f'herb {t2(fm.w_herb):.2f}, woody {t2(fm.w_woody):.2f}  t/ac')
    warn = ''
    if r.wind_limited:
        warn = ('\n> ⚠️ **Wind-limited**: midflame wind exceeded the Rothermel '
                '$0.9\\,I_R$ reliability limit and was capped. '
                'Uncheck the wind limit to see the raw value.')

    md = rf'''
### {fm.code} - {fm.name}  {'(dynamic)' if fm.dynamic else '(static)'}

| Rate of spread | |
|---|---|
| **{r.ros_m_min:.2f} m/min** | {r.ros_km_h:6.3f} km/h |
| {r.ros_m_s:8.4f} m/s | {r.ros_ft_min:8.2f} ft/min |
| {r.ros_ft_min/1.1:8.1f} chains/h | |

*{fm.description}*

**Fuel bed** loads: {loads} &nbsp;|&nbsp; depth {fm.depth} ft &nbsp;|&nbsp; dead Mx {fm.mx_dead*100:.0f}%

**Wind** midflame {midflame_ms:.2f} m/s ({wind_ftmin:.0f} ft/min) &nbsp;|&nbsp; {_waf_summary(waf_info)}
{_waf_block(waf_info)}
| Intermediate | imperial | SI (J, m, s) |
|---|---:|---:|
| Reaction intensity I_R | {r.reaction_intensity:,.0f} BTU/ft²/min | {r.reaction_intensity*IR_BTUFT2MIN_TO_KWM2:,.1f} kW/m² |
| Characteristic SAV σ | {r.characteristic_sav:,.0f} ft⁻¹ | {r.characteristic_sav*SAV_PERFT_TO_PERM:,.0f} m⁻¹ |
| Packing ratio β | {r.packing_ratio:.5f} | {r.packing_ratio:.5f} (-) |
| Relative packing ratio β_rel | {r.relative_packing_ratio:.2f} | {r.relative_packing_ratio:.2f} (-) |
| Propagating flux ratio ξ | {r.propagating_flux_ratio:.4f} | {r.propagating_flux_ratio:.4f} (-) |
| Wind factor φ_w | {r.wind_factor:.2f} | {r.wind_factor:.2f} (-) |
| Slope factor φ_s | {r.slope_factor:.2f} | {r.slope_factor:.2f} (-) |
| Heat sink ρ_b·ε·Q_ig | {r.heat_sink:.1f} BTU/ft³ | {r.heat_sink*HEAT_BTUFT3_TO_KJM3:,.0f} kJ/m³ |
| No-wind/no-slope ROS R_0 = I_R·ξ/(ρ_b·ε·Q_ig) | {ros_0_ft_min:.2f} ft/min | {ros_0_m_s:.3f} m/s |
| Live moisture of extinction Mx,live | {r.live_mx*100:.0f}% | {r.live_mx*100:.0f}% |
| Herb cured fraction | {r.cured_fraction:.2f} | {r.cured_fraction:.2f} (-) |

**Rate of spread** combines the rows above (Rothermel 1972, eq. 52):

$$\begin{{aligned}}
R &= \frac{{I_R\,\xi\,(1 + \phi_w + \phi_s)}}{{\rho_b\,\varepsilon\,Q_{{ig}}}} \\
  &= \frac{{{r.reaction_intensity:,.0f}\,\cdot\,{r.propagating_flux_ratio:.4f}\,\cdot\,(1 + {r.wind_factor:.2f} + {r.slope_factor:.2f})}}{{{r.heat_sink:.1f}}} \\
  &= {r.ros_ft_min:.2f}\ \text{{ft/min}} \rightarrow {r.ros_m_s:.4f}\ \text{{m/s}}
\end{{aligned}}$$

The numerator $I_R\,\xi\,(1+\phi_w+\phi_s)$ is the heat flux propagating into the unburnt fuel: the reaction intensity $I_R$ scaled by the no-wind, no-slope propagating flux ratio $\xi$, then amplified by the wind ($\phi_w$) and slope ($\phi_s$) factors. The denominator $\rho_b\,\varepsilon\,Q_{{ig}}$ is the heat sink, the energy needed to bring that fuel to ignition. Their ratio is the rate of spread shown at the top.

**Fuel burnout / heat release** (LES coupling, `burnout.py`) — after the front passes, the dry fuel decays as $F(t)=e^{{-(t-t_i)/T_f}}$ with the Anderson residence time $T_f=384/\sigma$. Because $I_R$ already carries Rothermel's moisture and mineral damping, it *is* the **sensible** heat-release rate to the atmosphere, so the **peak sensible flux equals $I_R$** (sensible energy $E_{{\text{{sens}}}}=I_R\cdot T_f$). The fuel water leaves as an **additional latent** flux on top — scaled by the gross-chemistry sensible:latent ratio so it tracks $I_R$ — and the **total** flux is their sum. Everything scales with $I_R$, so the block vanishes above the dead moisture of extinction, consistent with $\text{{ROS}}=0$ there.

| Burnout | value |
|---|---:|
| Anderson residence time T_f = 384/σ | {b.t_f:,.1f} s |
| Consumed dry load (full bed) | {b.load_dry:.2f} kg/m² |
| Fuel water vaporised | {water_vap_kg:.3f} kg/m² |
| Sensible heat E_sens = I_R·T_f | {b.energy_net_MJ:,.2f} MJ/m² |
| Latent heat (fuel-water vapour) | {b.energy_latent/1e6:,.2f} MJ/m² |
| Total heat release E_sens + E_lat | {b.energy_gross/1e6:,.2f} MJ/m² |
| Peak sensible flux = I_R (E_sens/T_f) | {peak_sensible_kW:,.0f} kW/m² |
| Peak latent flux E_lat/T_f | {peak_latent_kW:,.0f} kW/m² |
| Peak total flux (sensible + latent) | {peak_total_kW:,.0f} kW/m² |

**Wind factor**

$$\phi_w = C\,U^B\,\beta_{{\text{{rel}}}}^{{-E}}$$

(Rothermel 1972; coefficients fit with $\sigma$ in ft $^{{-1}}$ and $U$ in ft/min.)

| φ_w term | imperial (U in ft/min) | SI (U in m/s) |
|---|---:|---:|
| Characteristic SAV σ | {sigma:,.0f} ft⁻¹ | {sigma*SAV_PERFT_TO_PERM:,.0f} m⁻¹ |
| Midflame wind U | {wind_ftmin:,.0f} ft/min | {midflame_ms:.3f} m/s |
| C = 7.47·exp(-0.133·σ^0.55) | {c_w:.4f} | {c_w:.4f} |
| B = 0.02526·σ^0.54 | {b_w:.4f} | {b_w:.4f} (-) |
| E = 0.715·exp(-3.59×10⁻⁴·σ) | {e_w:.4f} | {e_w:.4f} (-) |
| K = C·β_rel^(-E) | {k_imp:.4f} | {k_si:.4f} = K·196.85^B ({u_factor:.2f}) |
| Wind factor φ_w | {r.wind_factor:.3f} | {r.wind_factor:.3f} (-) |

*$B$ and $E$ are dimensionless and unchanged between systems. Collecting the $\sigma$/packing terms gives $\phi_w = K\,U^B$ with $K = C\,\beta_{{\text{{rel}}}}^{{-E}}$. Only $K$ carries the wind-speed unit: because $U$ appears as $U^B$, switching $U$ from ft/min to m/s multiplies $K$ by $196.85^B$, so $\phi_w = K_{{\text{{SI}}}}\,U[\text{{m/s}}]^B$ reproduces the same value while $C$ keeps its original value.*
{warn}
'''
    with out:
        out.clear_output(wait=True)
        display(Markdown(md))

_controls = (fuel_dd, w_1h, w_10h, w_100h, w_herb, w_woody,
             m_1h, m_10h, m_100h, m_herb, m_woody, wind, wind_unit,
             wind_ref, slope, wind_limit, waf_mode, canopy_cover, canopy_height,
             crown_ratio, waf_direct)
for w in _controls:
    w.observe(_render, names='value')

_render()
display(ui, out)

## Notes

* **Midflame wind** is not the 10 m weather-station wind. Set the *wind reference* and a
  *WAF model* to convert 10-m or 20-ft wind to midflame wind (Albini & Baughman 1979;
  Andrews 2012, RMRS-GTR-266): unsheltered uses fuel-bed depth; sheltered uses canopy
  cover, height, and crown ratio (crown-fill portion $f = (CC/3)\cdot CR$, sheltered when
  $f > 5\%$). 10-m open wind is taken as $1.15\times$ the 20-ft wind.
* The **wind limit** caps the wind factor at the wind speed where Rothermel's data became
  unreliable ($0.9\,I_R$). Cruz & Finney (2013) later revised this; toggle it off to compare.
* **Dynamic models** (all GR/GS, SH1, SH9, TU1, TU3) transfer part of the live herbaceous
  load to dead as it cures: fully green at $\geq 120\%$ moisture, fully cured at $\leq 30\%$.
* `compute_ros` returns the full `SpreadResult`; use it directly for batch runs or plots.

## Fuel burnout time scale (LES coupling)

The **Fuel burnout / heat release** block in the result panel above is produced by
`burnout.py`. Rothermel itself is **quasi-steady**: it returns a rate of spread and a
reaction intensity $I_R$, but it has no clock — it never tells you how long a point
keeps burning after the front passes. An atmospheric LES needs exactly that: a
time-resolved surface **heat flux** as its lower boundary condition. WRF-SFIRE supplies
it with a separate fuel mass-loss model layered on top of the spread model; this is that
layer.

For a cell that ignites at time $t_i$, the remaining dry-fuel fraction decays
exponentially with an e-folding (burnout) time $T_f$:

$$F(t) = \exp\!\left(-\frac{t - t_i}{T_f}\right), \qquad Q(t) = \frac{E}{T_f}\,\exp\!\left(-\frac{t - t_i}{T_f}\right) \;\;[\text{W/m}^2]$$

The burnout time scale $T_f$ is **not** in Rothermel. The default closure here is the
Anderson (1969) flame residence time $\tau = 384/\sigma$ minutes, with $\sigma$ the
characteristic SAV ($\text{ft}^{-1}$) taken straight from the `SpreadResult` — finer fuel
burns faster, the same SAV dependence Rothermel uses internally.

**Tying the heat release to $I_R$ (energy-consistent).** $I_R$ already carries Rothermel's
moisture and mineral damping ($\eta_M,\eta_s$), so it *is* the **sensible** heat-release
rate delivered to the atmosphere. The sensible energy per unit ground area is Byram's
heat-per-area, $E_{\text{sens}} = I_R\,T_f$, so the **peak sensible flux is exactly $I_R$**
($E_{\text{sens}}/T_f = I_R$). The fuel moisture leaves as an **additional latent** flux on
top.

**How $E_{\text{lat}}$ is computed.** The latent energy is *not* simply the water's heat of
vaporisation — that would be a fixed offset that fails to vanish with $I_R$. Instead it is
$E_{\text{sens}}$ scaled by the fuel's **gross-chemistry latent:sensible ratio**, so it tracks
$I_R$. From the raw (undamped) chemistry of the consumed fuel, the full-load latent and
sensible parts are

$$E_{\text{lat}}^{\text{chem}} = m_w\,L_v, \qquad
E_{\text{sens}}^{\text{chem}} = m_{\text{dry}}\,h_c - m_w\,L_v,$$

with $m_w$ the consumed fuel-water mass [kg/m²], $L_v = 2.26\times10^{6}$ J/kg the latent heat
of vaporisation, $m_{\text{dry}}$ the consumed oven-dry load [kg/m²], and $h_c$ the fuel low
heat of combustion (8000 BTU/lb $\approx 1.86\times10^{7}$ J/kg). The latent flux delivered with
the moisture-damped sensible release is then their ratio applied to $E_{\text{sens}}$:

$$E_{\text{lat}} = E_{\text{sens}}\,\frac{E_{\text{lat}}^{\text{chem}}}{E_{\text{sens}}^{\text{chem}}}
= E_{\text{sens}}\,\frac{m_w\,L_v}{m_{\text{dry}}\,h_c - m_w\,L_v}.$$

If the water demand exceeds the chemistry's sensible release ($E_{\text{sens}}^{\text{chem}}\le 0$),
$E_{\text{lat}}$ is set to $0$. The **total** surface flux is the sum,
$\dot Q_{\text{tot}} = I_R + E_{\text{lat}}/T_f$. Nothing is double-counted: the latent term is
energy carried away by water vapour, kept separate from the already moisture-damped
sensible release. Because everything scales with $I_R$, when $I_R\to 0$ above the dead
moisture of extinction the sensible, latent, and total fluxes all vanish — exactly where
the spread model returns $\text{ROS}=0$. Raising any moisture lowers $I_R$ and hence all
three fluxes.

`compute_burnout` also keeps a **WRF-SFIRE mass-loss** basis (call it without
`reaction_intensity`): there the gross release is the full consumed dry load $\times$ heat
of combustion, with the evaporative sink subtracted to leave the sensible heat. That
counts the entire fuel load regardless of whether the front can sustain itself, so it
runs hotter than $I_R$ — by ~2× for grass and up to ~10× for heavy litter — and does
**not** vanish above the moisture of extinction.